# Лабораторная работа № 2
## Машинное обучение на больших данных с использованием Apache Spark и SparkML

### Часть 1. Регрессия GBTRegressor

Датасет Steam Games из лабораторной № 1. Источник данных — подготовленный Parquet `steam_games_cleaned.parquet`. Целевая переменная — `price` (цена игры). Идентификаторы, название, текстовые списки и признаки, непосредственно вычисленные из отзывов, исключены из признаков.

### Запуск Spark и загрузка Parquet

In [1]:
import os
from pyspark import SparkConf
from pyspark.sql import SparkSession, DataFrame
from pyspark.sql import functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler, Imputer, StandardScaler
from pyspark.ml.regression import GBTRegressor
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.feature import StringIndexer, OneHotEncoder
from pyspark.sql import functions as F
conf = (SparkConf().setAppName("Lab2 Steam GBT Regression")
        .setMaster("local[2]")
        .set("spark.driver.memory", "4g")
        .set("spark.sql.shuffle.partitions", "8"))
spark = SparkSession.builder.config(conf=conf).getOrCreate()
spark.sparkContext.setLogLevel("WARN")

In [2]:
# Если Parquet лежит в другой папке, укажите полный путь.
PARQUET_PATH = "steam_games_cleaned.parquet"
df = spark.read.parquet(PARQUET_PATH)
df.printSchema()
print("Количество строк:", df.count())
df.show(5, truncate=60)

root
 |-- app_id: string (nullable = true)
 |-- name: string (nullable = true)
 |-- release_date: string (nullable = true)
 |-- price: double (nullable = true)
 |-- estimated_owners: string (nullable = true)
 |-- positive: double (nullable = true)
 |-- negative: double (nullable = true)
 |-- recommendations: double (nullable = true)
 |-- average_playtime_forever: double (nullable = true)
 |-- steam_store_available: boolean (nullable = true)
 |-- steam_spy_available: boolean (nullable = true)
 |-- release_year: integer (nullable = true)
 |-- num_genres: double (nullable = true)
 |-- num_categories: double (nullable = true)
 |-- genres: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- developers: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- publishers: array (nullable = true)
 |    |-- element: string (containsNull = true)
 |-- categories: array (nullable = true)
 |    |-- element: string (containsNull = true)

Количество строк:

### Постановка задачи и подготовка данных

Предсказываем цену игры по году выпуска, числу жанров и категорий, положительным/отрицательным отзывам, рекомендациям и времени игры. Денежная переменная имеет длинный хвост, поэтому качество оценивается по RMSE, MAE и R². Пропуски числовых признаков заполняются медианой через `Imputer`; строки с отсутствующей целевой переменной удаляются.

In [3]:
# Приводим используемые поля к числовому типу; отсутствующие поля выявляются заранее.
numeric_candidates = ["release_year", "num_genres", "num_categories", "positive",
                      "negative", "recommendations", "average_playtime_forever"]
available = set(df.columns)
numeric_features = [c for c in numeric_candidates if c in available]
required = ["price"] + numeric_features
missing_columns = [c for c in required if c not in available]
if missing_columns:
    raise ValueError(f"В Parquet отсутствуют нужные столбцы: {missing_columns}. Проверьте файл после ЛР1.")

for c in required:
    df = df.withColumn(c, F.col(c).cast("double"))
df = df.filter(F.col("price").isNotNull() & (~F.isnan("price")) & (F.col("price") >= 0))
for c in numeric_features:
    df = df.withColumn(c, F.when(F.isnan(c), None).otherwise(F.col(c)))
print("Строк после очистки:", df.count())
df.select("price", *numeric_features).describe().show()

Строк после очистки: 129587
+-------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------------+
|summary|            price|      release_year|        num_genres|    num_categories|          positive|          negative|   recommendations|average_playtime_forever|
+-------+-----------------+------------------+------------------+------------------+------------------+------------------+------------------+------------------------+
|  count|           129587|            129587|            129587|            129587|            129587|            129587|            129587|                  129587|
|   mean|9.550480224023621|2022.0152252926605|3.5463433832097357|6.2281401683810875| 980.3353885806447|158.62392832614384|1220.8366337161808|       82.17552686612083|
| stddev|424.8580368852455| 3.442587851175953|2.0748283872062356|5.5323513748469235|27301.961115270686| 5222.196325441845|22258.836307884

### Выделение CSV для следующей лабораторной и разделение выборки

Случайная выборка сохраняется локально в CSV. При большом количестве строк она ограничивается 1000 наблюдениями; это небольшой вспомогательный набор, а не обучающая выборка.

In [4]:
sample_path = "steam_games_lab2_sample.csv"

sample = df.orderBy(F.rand(seed=11)).limit(1000)

# CSV не поддерживает массивы, поэтому преобразуем genres в строку
#if "genres" in sample.columns:
#    sample = sample.withColumn(
#        "genres",
#        F.concat_ws(", ", F.col("genres"))
#    )
sample_csv = (
    sample
    .withColumn("genres", F.concat_ws(", ", F.col("genres")))
    .withColumn("developers", F.concat_ws(", ", F.col("developers")))
    .withColumn("publishers", F.concat_ws(", ", F.col("publishers")))
    .withColumn("categories", F.concat_ws(", ", F.col("categories")))
)
sample_csv.coalesce(1).write \
    .mode("overwrite") \
    .option("header", True) \
    .csv(sample_path)

print(f"CSV сохранён в каталоге {sample_path}")

train_df, test_df = df.randomSplit([0.8, 0.2], seed=11)

CSV сохранён в каталоге steam_games_lab2_sample.csv


### ML-конвейер и подбор гиперпараметров

`VectorAssembler` объединяет числовые признаки, `Imputer` заполняет пропуски. GBTRegressor обучается с небольшим перебором глубины и числа итераций; число фолдов — 3, чтобы ограничить время вычислений.

In [5]:
imputed_features = [f"{c}_imputed" for c in numeric_features]
imputer = Imputer(inputCols=numeric_features, outputCols=imputed_features, strategy="median")
assembler = VectorAssembler(inputCols=imputed_features, outputCol="features", handleInvalid="skip")
gbt = GBTRegressor(featuresCol="features", labelCol="price", seed=11)
pipeline = Pipeline(stages=[imputer, assembler, gbt])

param_grid = (ParamGridBuilder()
              .addGrid(gbt.maxDepth, [3, 5])
              .addGrid(gbt.maxIter, [20, 40])
              .build())
evaluator = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName="rmse")
cv = CrossValidator(estimator=pipeline, estimatorParamMaps=param_grid,
                    evaluator=evaluator, numFolds=3, parallelism=2, seed=11)
cv_model = cv.fit(train_df)
print("Лучшие параметры:", cv_model.bestModel.stages[-1].extractParamMap())

Лучшие параметры: {Param(parent='GBTRegressor_c2e0c8fa45d9', name='cacheNodeIds', doc='If false, the algorithm will pass trees to executors to match instances with nodes. If true, the algorithm will cache node IDs for each instance. Caching can speed up training of deeper trees. Users can set how often should the cache be checkpointed or disable it by setting checkpointInterval.'): False, Param(parent='GBTRegressor_c2e0c8fa45d9', name='checkpointInterval', doc='set checkpoint interval (>= 1) or disable checkpoint (-1). E.g. 10 means that the cache will get checkpointed every 10 iterations. Note: this setting will be ignored if the checkpoint directory is not set in the SparkContext.'): 10, Param(parent='GBTRegressor_c2e0c8fa45d9', name='featureSubsetStrategy', doc="The number of features to consider for splits at each tree node. Supported options: 'auto' (choose automatically for task: If numTrees == 1, set to 'all'. If numTrees > 1 (forest), set to 'sqrt' for classification and to 'on

### Оценка на тестовой выборке

In [6]:
predictions = cv_model.transform(test_df).cache()
for metric in ["rmse", "mae", "r2"]:
    ev = RegressionEvaluator(labelCol="price", predictionCol="prediction", metricName=metric)
    print(f"Test {metric.upper()}: {ev.evaluate(predictions):.4f}")
predictions.select("price", "prediction").show(10)

Test RMSE: 25.0525
Test MAE: 8.4616
Test R2: -1.9562
+------------------+------------------+
|             price|        prediction|
+------------------+------------------+
|13.989999771118164| 5.624526999752045|
|               0.0|6.4409779518687955|
|11.989999771118164|13.411913231497895|
|0.9900000095367432| 5.336642224405194|
|14.989999771118164| 6.577829581809997|
|1.9900000095367432| 5.624526999752045|
|19.989999771118164| 7.081658588807768|
|               0.0| 5.624526999752045|
|              9.25| 5.375644718323857|
| 4.989999771118164| 6.577829581809997|
+------------------+------------------+
only showing top 10 rows



### Сохранение модели

Модель сохраняется в локальную папку Parquet-проекта. В следующей лабораторной её можно загрузить через `PipelineModel.load`. При необходимости укажите собственный путь.

In [7]:
MODEL_PATH = "steam_gbt_regression_model"
cv_model.bestModel.write().overwrite().save(MODEL_PATH)
print("Модель сохранена:", MODEL_PATH)
train_df.unpersist()
test_df.unpersist()
predictions.unpersist()
spark.stop()

Модель сохранена: steam_gbt_regression_model
